In [13]:
import pandas as pd
import os
from pathlib import Path

os.chdir('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity')

# dynamicProperties

Checking dynamic properties

In [14]:
# Load the parquet file
mof_df = pd.read_parquet("All_data/merged_all_filtered.parquet")

# Get unique dynamicProperties entries (drop NaN) and save to CSV
unique_dynamicProperties = (
    mof_df["dynamicProperties"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

unique_dynamicProperties.to_csv("All_data/dynamicProperties_unique.csv", index=False, header=["dynamicProperties"])

print(f"Saved {len(unique_dynamicProperties)} unique dynamicProperties entries to All_data/dynamicProperties_unique.csv")

Saved 8035 unique dynamicProperties entries to All_data/dynamicProperties_unique.csv


## individualCount

In [15]:
# check correspondence between dynamicProperties containing "observedindividualcount=" and individualCount
merged = pd.read_parquet("All_data/merged_all_filtered.parquet")
dp = merged["dynamicProperties"].astype(str)
ic = merged["individualCount"]

dp_mask = dp.str.contains("observedindividualcount=", case=False, na=False)
ic_mask = ic.notna() & (ic != -999)

print("Rows with 'observedindividualcount=' in dynamicProperties:", int(dp_mask.sum()))
print("Rows with individualCount present (not NA and not -999):", int(ic_mask.sum()))

print("\nContingency table (dp_has_obs_indiv x individualCount_present):")
print(pd.crosstab(dp_mask, ic_mask))

# examples where dp has observedindividualcount but individualCount is missing
print("\nExamples where dynamicProperties contains observedindividualcount but individualCount is missing:")
print(merged.loc[dp_mask & ~ic_mask, ["dynamicProperties", "individualCount"]].head(10).to_string(index=False))

# examples where individualCount present but dp does NOT contain observedindividualcount
print("\nExamples where individualCount present but dynamicProperties does NOT contain observedindividualcount:")
print(merged.loc[~dp_mask & ic_mask, ["dynamicProperties", "individualCount"]].head(10).to_string(index=False))

# extract numeric observedindividualcount from dynamicProperties and compare when both present
extracted = dp.str.extract(r"(?i)observedindividualcount=([0-9]+(?:\.[0-9]+)?)")[0].astype(float)
both_mask = dp_mask & ic_mask & extracted.notna()
print(f"\nRows where both present and extracted value parsed: {int(both_mask.sum())}")

if both_mask.any():
    diff = (merged.loc[both_mask, "individualCount"].astype(float) - extracted[both_mask]).abs()
    equal_count = int((diff == 0).sum())
    diff_count = int((diff != 0).sum())
    print(f" - exact matches between individualCount and extracted observedindividualcount: {equal_count}")
    print(f" - mismatches: {diff_count}")
    if diff_count:
        print("\nExamples of mismatches:")
        print(
            pd.DataFrame({
                "dynamicProperties": merged.loc[both_mask, "dynamicProperties"].str.slice(0,200),
                "individualCount": merged.loc[both_mask, "individualCount"],
                "extracted_observedindividualcount": extracted[both_mask]
            }).head(10).to_string(index=False)
        )

Rows with 'observedindividualcount=' in dynamicProperties: 23432
Rows with individualCount present (not NA and not -999): 2986321

Contingency table (dp_has_obs_indiv x individualCount_present):
individualCount     False    True 
dynamicProperties                 
False              951157  2977069
True                14180     9252

Examples where dynamicProperties contains observedindividualcount but individualCount is missing:
                                                                dynamicProperties  individualCount
                  observedweight=0.03;temperature=9.22;observedindividualcount=8;              NaN
observedweight=0.01;samplesize=0.3;temperature=10.02;observedindividualcount=3.5;              NaN
                                                     observedindividualcount=147;              NaN
    observedweight=0;samplesize=0.25;temperature=-0.8;observedindividualcount=12;              NaN
                                                     observedindividual

In [16]:
# check correspondence between dynamicProperties containing "observedindividualcount=" and individualCount
dp = merged["dynamicProperties"].astype(str)
ic = merged["individualCount"]

dp_mask = dp.str.contains("SubSampleNumbers=", case=False, na=False)
ic_mask = ic.notna() & (ic != -999)

print("Rows with 'SubSampleNumbers=' in dynamicProperties:", int(dp_mask.sum()))
print("Rows with individualCount present (not NA and not -999):", int(ic_mask.sum()))

print("\nContingency table (dp_has_obs_indiv x individualCount_present):")
print(pd.crosstab(dp_mask, ic_mask))

# examples where dp has SubSampleNumbers but individualCount is missing
print("\nExamples where dynamicProperties contains SubSampleNumbers but individualCount is missing:")
print(merged.loc[dp_mask & ~ic_mask, ["dynamicProperties", "individualCount"]].head(10).to_string(index=False))

# examples where individualCount present but dp does NOT contain SubSampleNumbers
print("\nExamples where individualCount present but dynamicProperties does NOT contain SubSampleNumbers:")
print(merged.loc[~dp_mask & ic_mask, ["dynamicProperties", "individualCount"]].head(10).to_string(index=False))

# extract numeric SubSampleNumbers from dynamicProperties and compare when both present
extracted = dp.str.extract(r"(?i)SubSampleNumbers=([0-9]+(?:\.[0-9]+)?)")[0].astype(float)
both_mask = dp_mask & ic_mask & extracted.notna()
print(f"\nRows where both present and extracted value parsed: {int(both_mask.sum())}")

if both_mask.any():
    diff = (merged.loc[both_mask, "individualCount"].astype(float) - extracted[both_mask]).abs()
    equal_count = int((diff == 0).sum())
    diff_count = int((diff != 0).sum())
    print(f" - exact matches between individualCount and extracted SubSampleNumbers: {equal_count}")
    print(f" - mismatches: {diff_count}")
    if diff_count:
        print("\nExamples of mismatches:")
        print(
            pd.DataFrame({
                "dynamicProperties": merged.loc[both_mask, "dynamicProperties"].str.slice(0,200),
                "individualCount": merged.loc[both_mask, "individualCount"],
                "extracted_SubSampleNumbers": extracted[both_mask]
            }).head(10).to_string(index=False)
        )

Rows with 'SubSampleNumbers=' in dynamicProperties: 867
Rows with individualCount present (not NA and not -999): 2986321

Contingency table (dp_has_obs_indiv x individualCount_present):
individualCount     False    True 
dynamicProperties                 
False              965224  2985567
True                  113      754

Examples where dynamicProperties contains SubSampleNumbers but individualCount is missing:
                                                                                                                                                                                        dynamicProperties  individualCount
                 SubSampleWeightInGrams=0.002; SubSampleNumbers=; CalculatedWeightInGrams=0.002; CalculatedNumbers=; SampledAll=Y; AverageWeightInGrams=0.0005; NonSubSubWeightInGrams=; NonSubSubNumber=              NaN
          SubSampleWeightInGrams=0.041999999; SubSampleNumbers=; CalculatedWeightInGrams=0.041999999; CalculatedNumbers=; SampledAll=Y; AverageW

In [17]:
# Fill individualCount from dynamicProperties when individualCount is NaN
merged_all_mof = merged.copy()

observed_ic = (
    merged_all_mof["dynamicProperties"]
    .astype("string")
    .str.extract(r"(?i)observedindividualcount=([0-9]+(?:\.[0-9]+)?)")[0]
    .astype(float)
)

fill_mask = merged_all_mof["individualCount"].isna() & observed_ic.notna()

merged_all_mof.loc[fill_mask, "individualCount"] = observed_ic[fill_mask]

print(f"Filled {int(fill_mask.sum())} individualCount values from observedindividualcount.")

preview_cols = ["dynamicProperties", "individualCount"]
print(merged_all_mof.loc[fill_mask, preview_cols].head(10).to_string(index=False))

merged_all_mof.to_parquet("merged_all_mof.parquet", index=False)
print("Saved merged_all_mof.parquet")

Filled 14180 individualCount values from observedindividualcount.
                                                                dynamicProperties  individualCount
                  observedweight=0.03;temperature=9.22;observedindividualcount=8;              8.0
observedweight=0.01;samplesize=0.3;temperature=10.02;observedindividualcount=3.5;              3.5
                                                     observedindividualcount=147;            147.0
    observedweight=0;samplesize=0.25;temperature=-0.8;observedindividualcount=12;             12.0
                                                     observedindividualcount=147;            147.0
                  observedweight=0.07;temperature=9.22;observedindividualcount=4;              4.0
                                                       observedindividualcount=4;              4.0
                                                       observedindividualcount=3;              3.0
                 observedweight=0.07;temper

In [18]:
# Fill individualCount from dynamicProperties when individualCount is NaN
merged_all_mof = merged.copy()

observed_ic = (
    merged_all_mof["dynamicProperties"]
    .astype("string")
    .str.extract(r"(?i)SubSampleNumbers=([0-9]+(?:\.[0-9]+)?)")[0]
    .astype(float)
)

fill_mask = merged_all_mof["individualCount"].isna() & observed_ic.notna()

merged_all_mof.loc[fill_mask, "individualCount"] = observed_ic[fill_mask]

print(f"Filled {int(fill_mask.sum())} individualCount values from SubSampleNumbers.")

preview_cols = ["dynamicProperties", "individualCount"]
print(merged_all_mof.loc[fill_mask, preview_cols].head(10).to_string(index=False))

merged_all_mof.to_parquet("merged_all_mof.parquet", index=False)
print("Saved merged_all_mof.parquet")

Filled 0 individualCount values from SubSampleNumbers.
Empty DataFrame
Columns: [dynamicProperties, individualCount]
Index: []
Saved merged_all_mof.parquet


## biomass

In [19]:
# Fill biomass from observedweight in dynamicProperties when biomass is empty

mof_df = pd.read_parquet("merged_all_mof.parquet")

biomass_empty = mof_df["biomass"].isna() | (mof_df["biomass"].astype("string").str.strip() == "")
observed_weight = (
    mof_df["dynamicProperties"]
    .astype("string")
    .str.extract(r"(?i)observedweight=([0-9]+(?:\.[0-9]+)?)")[0]
    .astype(float)
)

fill_mask = biomass_empty & observed_weight.notna()
mof_df.loc[fill_mask, "biomass"] = observed_weight[fill_mask]

print(f"Filled {int(fill_mask.sum())} biomass values from observedweight.")

preview_cols = ["dynamicProperties", "biomass"]
print(mof_df.loc[fill_mask, preview_cols].head(10).to_string(index=False))

mof_df.to_parquet("merged_all_mof.parquet", index=False)
print("Saved merged_all_mof.parquet")

Filled 4896 biomass values from observedweight.
                                                                 dynamicProperties  biomass
                   observedweight=0.03;temperature=9.22;observedindividualcount=8;     0.03
 observedweight=0.01;samplesize=0.3;temperature=10.02;observedindividualcount=3.5;     0.01
     observedweight=0;samplesize=0.25;temperature=-0.8;observedindividualcount=12;     0.00
                   observedweight=0.07;temperature=9.22;observedindividualcount=4;     0.07
                  observedweight=0.07;temperature=6.9;observedindividualcount=182;     0.07
     observedweight=0;samplesize=0.26;temperature=-1.57;observedindividualcount=4;     0.00
 observedweight=0.01;samplesize=0.26;temperature=-1.58;observedindividualcount=36;     0.01
observedweight=0.04;samplesize=0.26;temperature=-1.44;observedindividualcount=616;     0.04
 observedweight=0.02;samplesize=0.39;temperature=-1.1;observedindividualcount=2.5;     0.02
 observedweight=0.02;samplesize=

In [20]:
# Fill biomass from observedweight in dynamicProperties when biomass is empty

mof_df = pd.read_parquet("merged_all_mof.parquet")

biomass_empty = mof_df["biomass"].isna() | (mof_df["biomass"].astype("string").str.strip() == "")
observed_weight = (
    mof_df["dynamicProperties"]
    .astype("string")
    .str.extract(r"(?i)SubSampleWeightInGrams=([0-9]+(?:\.[0-9]+)?)")[0]
    .astype(float)
)

fill_mask = biomass_empty & observed_weight.notna()
mof_df.loc[fill_mask, "biomass"] = observed_weight[fill_mask]
mof_df.loc[fill_mask, "biomassUnit"] = "g"

print(f"Filled {int(fill_mask.sum())} biomass values from SubSampleWeightInGrams.")

preview_cols = ["dynamicProperties", "biomass"]
print(mof_df.loc[fill_mask, preview_cols].head(10).to_string(index=False))

mof_df.to_parquet("merged_all_mof.parquet", index=False)
print("Saved merged_all_mof.parquet")

Filled 867 biomass values from SubSampleWeightInGrams.
                                                                                                                                                                                            dynamicProperties  biomass
   SubSampleWeightInGrams=0.048999999; SubSampleNumbers=51; CalculatedWeightInGrams=0.282999992; CalculatedNumbers=300; SampledAll=N; AverageWeightInGrams=0.00096; NonSubSubWeightInGrams=; NonSubSubNumber=    0.049
                    SubSampleWeightInGrams=0.001; SubSampleNumbers=1; CalculatedWeightInGrams=0.001; CalculatedNumbers=1; SampledAll=N; AverageWeightInGrams=0.001; NonSubSubWeightInGrams=; NonSubSubNumber=    0.001
SubSampleWeightInGrams=0.037999999; SubSampleNumbers=11; CalculatedWeightInGrams=0.037999999; CalculatedNumbers=11; SampledAll=N; AverageWeightInGrams=0.003454545; NonSubSubWeightInGrams=; NonSubSubNumber=    0.038
                   SubSampleWeightInGrams=0.014; SubSampleNumbers=4; CalculatedWeight

In [21]:
# Fill biomass from observedweight in dynamicProperties when biomass is empty

mof_df = pd.read_parquet("merged_all_mof.parquet")

biomass_empty = mof_df["biomass"].isna() | (mof_df["biomass"].astype("string").str.strip() == "")
observed_weight = (
    mof_df["dynamicProperties"]
    .astype("string")
    .str.extract(r"(?i)ObservedWeight=([0-9]+(?:\.[0-9]+)?)")[0]
    .astype(float)
)

fill_mask = biomass_empty & observed_weight.notna()
mof_df.loc[fill_mask, "biomass"] = observed_weight[fill_mask]

print(f"Filled {int(fill_mask.sum())} biomass values from ObservedWeight.")

preview_cols = ["dynamicProperties", "biomass"]
print(mof_df.loc[fill_mask, preview_cols].head(10).to_string(index=False))

mof_df.to_parquet("merged_all_mof.parquet", index=False)
print("Saved merged_all_mof.parquet")

Filled 0 biomass values from ObservedWeight.
Empty DataFrame
Columns: [dynamicProperties, biomass]
Index: []
Saved merged_all_mof.parquet


In [22]:
# Fill biomass from observedweight in dynamicProperties when biomass is empty

mof_df = pd.read_parquet("merged_all_mof.parquet")

biomass_empty = mof_df["biomass"].isna() | (mof_df["biomass"].astype("string").str.strip() == "")
observed_weight = (
    mof_df["dynamicProperties"]
    .astype("string")
    .str.extract(r"(?i)Mass (kg)=([0-9]+(?:\.[0-9]+)?)")[0]
    .astype(float)
)

fill_mask = biomass_empty & observed_weight.notna()
mof_df.loc[fill_mask, "biomass"] = observed_weight[fill_mask]
mof_df.loc[fill_mask, "biomassUnit"] = "kg"

print(f"Filled {int(fill_mask.sum())} biomass values from Mass (kg).")

preview_cols = ["dynamicProperties", "biomass"]
print(mof_df.loc[fill_mask, preview_cols].head(10).to_string(index=False))

mof_df.to_parquet("merged_all_mof.parquet", index=False)
print("Saved merged_all_mof.parquet")

Filled 0 biomass values from Mass (kg).
Empty DataFrame
Columns: [dynamicProperties, biomass]
Index: []
Saved merged_all_mof.parquet


## areaSampled

In [23]:
# Fill areaSampled and areaSampledUnit from SampleSize in dynamicProperties when empty

mof_df = pd.read_parquet("merged_all_mof.parquet")

area_sampled_empty = mof_df["areaSampled"].isna() | (mof_df["areaSampled"].astype("string").str.strip() == "")
area_sampled_unit_empty = mof_df["areaSampledUnit"].isna() | (mof_df["areaSampledUnit"].astype("string").str.strip() == "")

sample_size_raw = (
    mof_df["dynamicProperties"]
    .astype("string")
    .str.extract(r'(?i)SampleSize=([^;\"]+)')[0]
    .str.strip()
    .str.strip('"')
)

sample_size_parts = sample_size_raw.str.extract(r'^\s*([0-9]+(?:[.,][0-9]+)?)\s*(.*)$')
sample_size_value = sample_size_parts[0].str.replace(',', '.', regex=False).astype(float)
sample_size_unit = sample_size_parts[1].fillna(sample_size_raw).str.strip()

fill_area_mask = area_sampled_empty & sample_size_value.notna()
fill_unit_mask = fill_area_mask & area_sampled_unit_empty & sample_size_unit.notna()

mof_df.loc[fill_area_mask, "areaSampled"] = sample_size_value[fill_area_mask]
mof_df.loc[fill_unit_mask, "areaSampledUnit"] = sample_size_unit[fill_unit_mask]

print(f"Filled {int(fill_area_mask.sum())} areaSampled values and {int(fill_unit_mask.sum())} areaSampledUnit values from SampleSize.")

preview_cols = ["dynamicProperties", "areaSampled", "areaSampledUnit"]
print(mof_df.loc[fill_area_mask | fill_unit_mask, preview_cols].head(30).to_string(index=False))

mof_df.to_parquet("merged_all_mof.parquet", index=False)
print("Saved merged_all_mof.parquet")

Filled 32670 areaSampled values and 32670 areaSampledUnit values from SampleSize.
                                                                 dynamicProperties  areaSampled areaSampledUnit
 observedweight=0.01;samplesize=0.3;temperature=10.02;observedindividualcount=3.5;         0.30                
     observedweight=0;samplesize=0.25;temperature=-0.8;observedindividualcount=12;         0.25                
                                                                  samplesize=0.75;         0.75                
                                                                   samplesize=2.3;         2.30                
                                                                  samplesize=4.97;         4.97                
     observedweight=0;samplesize=0.26;temperature=-1.57;observedindividualcount=4;         0.26                
 observedweight=0.01;samplesize=0.26;temperature=-1.58;observedindividualcount=36;         0.26                
observedweight=0.04;sa

In [24]:
unique_area_sampled_units = (
    mof_df["areaSampledUnit"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

print(f"Unique areaSampledUnit values: {len(unique_area_sampled_units)}")
print(unique_area_sampled_units.to_string(index=False))

Unique areaSampledUnit values: 4
   
cm²
 m2
 m²


In [25]:
mask = mof_df["areaSampledUnit"].astype(str).str.strip() == "Not available"
mof_df.loc[mask, "areaSampledUnit"] = None
print(f"Replaced {int(mask.sum())} 'Not available' entries in areaSampledUnit with NaN")
mof_df.to_parquet("merged_all_mof.parquet", index=False)

Replaced 0 'Not available' entries in areaSampledUnit with NaN


# OBIS MoF

In [ ]:
obis = pd.read_parquet('OBIS/mof_OBIS.parquet')
merged = pd.read_parquet('All_data/merged_all_mof.parquet')

obis_filtered = obis[obis['id'].isin(merged['sourceID'])].copy()

removed_count = len(obis) - len(obis_filtered)
print(f"Removed {removed_count} records")

obis_filtered.to_parquet('OBIS/mof_OBIS_filtered.parquet', index=False)

In [ ]:
# Load the parquet file
mof_df = pd.read_parquet("OBIS/mof_OBIS_filtered.parquet")

# Get unique measurementType + measurementUnit pairs and save to CSV
unique_measurement_pairs = (
    mof_df[["measurementType", "measurementUnit"]]
    .dropna(subset=["measurementType"])
    .drop_duplicates()
    .sort_values(["measurementType", "measurementUnit"], na_position="last")
    .reset_index(drop=True)
)

unique_measurement_pairs.to_csv(
    "OBIS/measurementType_unique.csv", index=False, header=["measurementType", "measurementUnit"]
)

print(
    f"Saved {len(unique_measurement_pairs)} unique measurementType/measurementUnit pairs to measurementType_unique.csv"
)

Saved 337 unique measurementType/measurementUnit pairs to measurementType_unique.csv


## individualCount

In [ ]:
# Fill individualCount in All_data/merged_all_mof.parquet from OBIS measurementValue
# when measurementType indicates abundance/count and individualCount is empty.

target_types = {
    "abundnr",
    "individualcount",
    "count",
    "count (in assayed sample) of biological entity specified elsewhere",
    "observedindividualcount",
    "taxon abundance",
}

obis_df = pd.read_parquet("OBIS/mof_OBIS_filtered.parquet")
merged_df = pd.read_parquet("merged_all_mof.parquet")

# Keep only relevant measurementType rows (case-insensitive)
obis_counts = obis_df[
    obis_df["measurementType"].astype("string").str.strip().str.lower().isin(target_types)
].copy()

# Parse measurementValue as numeric
obis_counts["measurementValue_num"] = pd.to_numeric(
    obis_counts["measurementValue"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)

# One value per id (first non-null numeric value)
id_to_count = (
    obis_counts.dropna(subset=["measurementValue_num"])
    .drop_duplicates(subset=["id"], keep="first")
    .set_index("id")["measurementValue_num"]
)

# Treat empty as NaN or -999
empty_ic_mask = merged_df["individualCount"].isna() | (merged_df["individualCount"] == -999)

# Map OBIS id -> merged sourceID
mapped_counts = merged_df["sourceID"].astype("string").map(id_to_count)

fill_mask = empty_ic_mask & mapped_counts.notna()
merged_df.loc[fill_mask, "individualCount"] = mapped_counts[fill_mask]

added_count = int(fill_mask.sum())
print(f"Added {added_count} individualCount values from OBIS measurementValue.")

merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)
print("Saved All_data/merged_all_mof.parquet")

Added 13847 individualCount values from OBIS measurementValue.
Saved All_data/merged_all_mof.parquet


In [ ]:
# Fill individualCount (and areaSampledUnit) in All_data/merged_all_mof.parquet
# from OBIS/mof_OBIS_filtered.parquet when measurementType indicates abundance.

target_types = {
    "abundance",
    "abundance of biological entity specified elsewhere per unit area of the bed",
    "abundance of biological entity specified elsewhere per unit volume of the water body",
    "abundance_[#/m3]",
}

obis_df = pd.read_parquet("OBIS/mof_OBIS_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

# Filter abundance-like measurement types (case-insensitive)
obis_abund = obis_df[
    obis_df["measurementType"].astype("string").str.strip().str.lower().isin(target_types)
].copy()

# Parse measurementValue as numeric for individualCount
obis_abund["measurementValue_num"] = pd.to_numeric(
    obis_abund["measurementValue"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)

# Keep one row per id (first valid numeric value)
id_map = (
    obis_abund.dropna(subset=["measurementValue_num"])
    .drop_duplicates(subset=["id"], keep="first")
    .set_index("id")
)

id_to_count = id_map["measurementValue_num"]
id_to_unit = id_map["measurementUnit"].astype("string")

# Empty masks
empty_ic_mask = merged_df["individualCount"].isna() | (merged_df["individualCount"] == -999)
empty_asu_mask = merged_df["areaSampledUnit"].isna() | (
    merged_df["areaSampledUnit"].astype("string").str.strip() == ""
)

# Map OBIS id -> merged sourceID
mapped_count = merged_df["sourceID"].astype("string").map(id_to_count)
mapped_unit = merged_df["sourceID"].astype("string").map(id_to_unit)

# Fill
fill_ic_mask = empty_ic_mask & mapped_count.notna()
fill_asu_mask = empty_asu_mask & mapped_unit.notna()

merged_df.loc[fill_ic_mask, "individualCount"] = mapped_count[fill_ic_mask]
merged_df.loc[fill_asu_mask, "areaSampledUnit"] = mapped_unit[fill_asu_mask]

print(f"Added {int(fill_ic_mask.sum())} individualCount values from OBIS abundance measurementValue.")
print(f"Added {int(fill_asu_mask.sum())} areaSampledUnit values from OBIS abundance measurementUnit.")

merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)
print("Saved All_data/merged_all_mof.parquet")

Added 79070 individualCount values from OBIS abundance measurementValue.
Added 79271 areaSampledUnit values from OBIS abundance measurementUnit.
Saved All_data/merged_all_mof.parquet


In [ ]:
# Fill individualCount, areaSampled, and areaSampledUnit from count_per_1000m2 in OBIS

obis_df = pd.read_parquet("OBIS/mof_OBIS_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

# Filter for count_per_1000m2 measurement type (case-insensitive)
obis_count_1000m2 = obis_df[
    obis_df["measurementType"].astype("string").str.strip().str.lower() == "count_per_1000m2"
].copy()

# Parse measurementValue as numeric
obis_count_1000m2["measurementValue_num"] = pd.to_numeric(
    obis_count_1000m2["measurementValue"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)

# Keep one row per id (first valid numeric value)
id_map = (
    obis_count_1000m2.dropna(subset=["measurementValue_num"])
    .drop_duplicates(subset=["id"], keep="first")
    .set_index("id")
)

id_to_count = id_map["measurementValue_num"]

# Empty masks
empty_ic_mask = merged_df["individualCount"].isna() | (merged_df["individualCount"] == -999)
empty_area_mask = merged_df["areaSampled"].isna() | (merged_df["areaSampled"].astype("string").str.strip() == "")
empty_unit_mask = merged_df["areaSampledUnit"].isna() | (merged_df["areaSampledUnit"].astype("string").str.strip() == "")

# Map OBIS id -> merged sourceID
mapped_count = merged_df["sourceID"].astype("string").map(id_to_count)

# Fill
fill_ic_mask = empty_ic_mask & mapped_count.notna()
fill_area_mask = empty_area_mask & mapped_count.notna()
fill_unit_mask = empty_unit_mask & mapped_count.notna()

merged_df.loc[fill_ic_mask, "individualCount"] = mapped_count[fill_ic_mask]
merged_df.loc[fill_area_mask, "areaSampled"] = 1000.0
merged_df.loc[fill_unit_mask, "areaSampledUnit"] = "m2"

added_count = int(fill_ic_mask.sum())
print(f"Added {added_count} entries from count_per_1000m2")
print(f" - {int(fill_ic_mask.sum())} individualCount values")
print(f" - {int(fill_area_mask.sum())} areaSampled values (set to 1000)")
print(f" - {int(fill_unit_mask.sum())} areaSampledUnit values (set to m2)")

merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)
print("Saved All_data/merged_all_mof.parquet")

Added 2331 entries from count_per_1000m2
 - 2331 individualCount values
 - 2331 areaSampled values (set to 1000)
 - 2331 areaSampledUnit values (set to m2)
Saved All_data/merged_all_mof.parquet


In [ ]:
# Fill individualCount and samplingEffort from OBIS for standardized tow measurements

obis_df = pd.read_parquet("OBIS/mof_OBIS_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

# Filter for standardized tow measurement types (case-insensitive)
target_types = {
    "standardized number of individuals captured for 24 min tow",
    "standardized number of individuals captured for 15 min tow",
}

obis_tow = obis_df[
    obis_df["measurementType"].astype("string").str.strip().str.lower().isin(target_types)
].copy()

# Parse measurementValue as numeric
obis_tow["measurementValue_num"] = pd.to_numeric(
    obis_tow["measurementValue"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)

# Keep one row per id (first valid numeric value)
id_map = (
    obis_tow.dropna(subset=["measurementValue_num"]) 
    .drop_duplicates(subset=["id"], keep="first")
    .set_index("id")
)

id_to_count = id_map["measurementValue_num"]
id_to_type = id_map["measurementType"].astype("string")

# Empty masks
empty_ic_mask = merged_df["individualCount"].isna() | (merged_df["individualCount"] == -999)
empty_effort_mask = merged_df["samplingEffort"].isna() | (
    merged_df["samplingEffort"].astype("string").str.strip() == ""
)

# Map OBIS id -> merged sourceID
mapped_count = merged_df["sourceID"].astype("string").map(id_to_count)
mapped_type = merged_df["sourceID"].astype("string").map(id_to_type)

# Fill
fill_ic_mask = empty_ic_mask & mapped_count.notna()
fill_effort_mask = empty_effort_mask & mapped_type.notna()

merged_df.loc[fill_ic_mask, "individualCount"] = mapped_count[fill_ic_mask]
merged_df.loc[fill_effort_mask, "samplingEffort"] = mapped_type[fill_effort_mask]

added_count = int(fill_ic_mask.sum())
added_effort = int(fill_effort_mask.sum())
print(f"Added {added_count} individualCount values from standardized tow measurements.")
print(f"Added {added_effort} samplingEffort values from standardized tow measurements.")

merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)
print("Saved All_data/merged_all_mof.parquet")

Added 1916 individualCount values from standardized tow measurements.
Added 0 samplingEffort values from standardized tow measurements.
Saved All_data/merged_all_mof.parquet


## biomass

In [ ]:
# Fill biomass and biomassUnit from OBIS measurementValue when biomass is empty

target_types = {
    "biomass",
    "mass",
    "total macrofaunal biomass",
    "wet weight biomass (in assayed sample) of biological entity specified elsewhere",
    "wet weight biomass of biological entity specified elsewhere per unit area of the bed",
    "weight_whole",
    "wet weight",
}

obis_df = pd.read_parquet("OBIS/mof_OBIS_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

# Filter biomass-like measurement types (case-insensitive)
obis_biomass = obis_df[
    obis_df["measurementType"].astype("string").str.strip().str.lower().isin(target_types)
].copy()

# Parse measurementValue as numeric
obis_biomass["measurementValue_num"] = pd.to_numeric(
    obis_biomass["measurementValue"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)

# Keep one row per id (first valid numeric value)
id_map = (
    obis_biomass.dropna(subset=["measurementValue_num"])
    .drop_duplicates(subset=["id"], keep="first")
    .set_index("id")
)

id_to_biomass = id_map["measurementValue_num"]
id_to_unit = id_map["measurementUnit"].astype("string")

# Empty masks
empty_biomass_mask = merged_df["biomass"].isna() | (merged_df["biomass"].astype("string").str.strip() == "")
empty_biomass_unit_mask = merged_df["biomassUnit"].isna() | (merged_df["biomassUnit"].astype("string").str.strip() == "")

# Map OBIS id -> merged sourceID
mapped_biomass = merged_df["sourceID"].astype("string").map(id_to_biomass)
mapped_unit = merged_df["sourceID"].astype("string").map(id_to_unit)

# Fill
fill_biomass_mask = empty_biomass_mask & mapped_biomass.notna()
fill_unit_mask = empty_biomass_unit_mask & mapped_unit.notna()

merged_df.loc[fill_biomass_mask, "biomass"] = mapped_biomass[fill_biomass_mask]
merged_df.loc[fill_unit_mask, "biomassUnit"] = mapped_unit[fill_unit_mask]

print(f"Added {int(fill_biomass_mask.sum())} biomass values from OBIS measurementValue.")
print(f"Added {int(fill_unit_mask.sum())} biomassUnit values from OBIS measurementUnit.")

merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)
print("Saved All_data/merged_all_mof.parquet")

Added 109248 biomass values from OBIS measurementValue.
Added 109248 biomassUnit values from OBIS measurementUnit.
Saved All_data/merged_all_mof.parquet


In [ ]:
# Fill biomass and biomassUnit from OBIS measurementValue for specific measurement types

target_types = {
    "weight_g_per_1000m2",
    "biomass_[mg dw/m3]",
    "polychaete biomass",
}

obis_df = pd.read_parquet("OBIS/mof_OBIS_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

# Filter for target measurement types (case-insensitive)
obis_weight = obis_df[
    obis_df["measurementType"].astype("string").str.strip().str.lower().isin(target_types)
].copy()

# Parse measurementValue as numeric
obis_weight["measurementValue_num"] = pd.to_numeric(
    obis_weight["measurementValue"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)

# Keep one row per id (first valid numeric value)
id_map = (
    obis_weight.dropna(subset=["measurementValue_num"])
    .drop_duplicates(subset=["id"], keep="first")
    .set_index("id")
)

id_to_biomass = id_map["measurementValue_num"]
id_to_type = id_map["measurementType"].astype("string").str.lower()

# Empty biomass mask
empty_biomass_mask = merged_df["biomass"].isna() | (merged_df["biomass"].astype("string").str.strip() == "")

# Map OBIS id -> merged sourceID
mapped_biomass = merged_df["sourceID"].astype("string").map(id_to_biomass)
mapped_type = merged_df["sourceID"].astype("string").map(id_to_type)

# Fill biomass
fill_biomass_mask = empty_biomass_mask & mapped_biomass.notna()
merged_df.loc[fill_biomass_mask, "biomass"] = mapped_biomass[fill_biomass_mask]

# Fill biomassUnit based on type
weight_g_mask = fill_biomass_mask & (mapped_type == "weight_g_per_1000m2")
biomass_mg_mask = fill_biomass_mask & (mapped_type == "biomass_[mg dw/m3]")
polychaete_mask = fill_biomass_mask & (mapped_type == "polychaete biomass")

merged_df.loc[weight_g_mask, "biomassUnit"] = "g/1000 m2"
merged_df.loc[biomass_mg_mask, "biomassUnit"] = "mg dw/m3"
merged_df.loc[polychaete_mask, "biomassUnit"] = "polychaete biomass (g)"

added_count = int(fill_biomass_mask.sum())
print(f"Added {added_count} biomass values from OBIS.")
print(f" - {int(weight_g_mask.sum())} entries from weight_g_per_1000m2 (unit: g/1000 m2)")
print(f" - {int(biomass_mg_mask.sum())} entries from Biomass_[mg dw/m3] (unit: mg dw/m3)")
print(f" - {int(polychaete_mask.sum())} entries from Polychaete biomass (unit: polychaete biomass (g))")

merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)
print("Saved All_data/merged_all_mof.parquet")

Added 4873 biomass values from OBIS.
 - 2331 entries from weight_g_per_1000m2 (unit: g/1000 m2)
 - 2542 entries from Biomass_[mg dw/m3] (unit: mg dw/m3)
 - 0 entries from Polychaete biomass (unit: polychaete biomass (g))
Saved All_data/merged_all_mof.parquet


In [ ]:
obis_df = pd.read_parquet("OBIS/mof_OBIS_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

target_types = {
    "standardized weight of total capture for 15 min tow",
    "standardized weight of total capture for 24 min tow",
}

obis_tow_weight = obis_df[
    obis_df["measurementType"].astype("string").str.strip().str.lower().isin(target_types)
].copy()

obis_tow_weight["measurementValue_num"] = pd.to_numeric(
    obis_tow_weight["measurementValue"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)

id_map = (
    obis_tow_weight.dropna(subset=["measurementValue_num"])
    .drop_duplicates(subset=["id"], keep="first")
    .set_index("id")
)

id_to_biomass = id_map["measurementValue_num"]
id_to_effort = id_map["measurementType"].astype("string")

empty_biomass_mask = merged_df["biomass"].isna() | (merged_df["biomass"].astype("string").str.strip() == "")
empty_effort_mask = merged_df["samplingEffort"].isna() | (merged_df["samplingEffort"].astype("string").str.strip() == "")

mapped_biomass = merged_df["sourceID"].astype("string").map(id_to_biomass)
mapped_effort = merged_df["sourceID"].astype("string").map(id_to_effort)

fill_biomass_mask = empty_biomass_mask & mapped_biomass.notna()
fill_effort_mask = empty_effort_mask & mapped_effort.notna()

merged_df.loc[fill_biomass_mask, "biomass"] = mapped_biomass[fill_biomass_mask]
# When biomass is added from OBIS standardized weight, set unit to kilograms
merged_df.loc[fill_biomass_mask, "biomassUnit"] = "kg"
merged_df.loc[fill_effort_mask, "samplingEffort"] = mapped_effort[fill_effort_mask]

print(f"Added {int(fill_biomass_mask.sum())} biomass values from standardized tow weight measurements.")
print(f"Added {int(fill_effort_mask.sum())} samplingEffort values from standardized tow weight measurements.")

merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)
print("Saved All_data/merged_all_mof.parquet")

Added 2269 biomass values from standardized tow weight measurements.
Added 0 samplingEffort values from standardized tow weight measurements.
Saved All_data/merged_all_mof.parquet


## End coordinates

In [ ]:
# Fill decimalLatitudeStop and decimalLongitudeStop from OBIS mof data

obis_df = pd.read_parquet("OBIS/mof_OBIS_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

# Filter for EndLatitude and EndLongitude
obis_end_lat = obis_df[obis_df["measurementType"] == "EndLatitude"].copy()
obis_end_lon = obis_df[obis_df["measurementType"] == "EndLongitude"].copy()

# Parse measurementValue as numeric
obis_end_lat["measurementValue_num"] = pd.to_numeric(
    obis_end_lat["measurementValue"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)
obis_end_lon["measurementValue_num"] = pd.to_numeric(
    obis_end_lon["measurementValue"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)

# Keep one row per id (first valid numeric value)
id_to_lat = (
    obis_end_lat.dropna(subset=["measurementValue_num"])
    .drop_duplicates(subset=["id"], keep="first")
    .set_index("id")["measurementValue_num"]
)

id_to_lon = (
    obis_end_lon.dropna(subset=["measurementValue_num"])
    .drop_duplicates(subset=["id"], keep="first")
    .set_index("id")["measurementValue_num"]
)

# Empty masks
empty_lat_mask = merged_df["decimalLatitudeStop"].isna()
empty_lon_mask = merged_df["decimalLongitudeStop"].isna()

# Map OBIS id -> merged sourceID
mapped_lat = merged_df["sourceID"].astype("string").map(id_to_lat)
mapped_lon = merged_df["sourceID"].astype("string").map(id_to_lon)

# Fill
fill_lat_mask = empty_lat_mask & mapped_lat.notna()
fill_lon_mask = empty_lon_mask & mapped_lon.notna()

merged_df.loc[fill_lat_mask, "decimalLatitudeStop"] = mapped_lat[fill_lat_mask]
merged_df.loc[fill_lon_mask, "decimalLongitudeStop"] = mapped_lon[fill_lon_mask]

print(f"Added {int(fill_lat_mask.sum())} decimalLatitudeStop values from OBIS EndLatitude.")
print(f"Added {int(fill_lon_mask.sum())} decimalLongitudeStop values from OBIS EndLongitude.")

merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)
print("Saved All_data/merged_all_mof.parquet")

Added 8913 decimalLatitudeStop values from OBIS EndLatitude.
Added 8913 decimalLongitudeStop values from OBIS EndLongitude.
Saved All_data/merged_all_mof.parquet


## Gear

In [ ]:
# Fill gear in All_data/merged_all_mof.parquet from OBIS measurementValue for gear-like measurementType
obis_df = pd.read_parquet("OBIS/mof_OBIS_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

targets = {
    "equipment",
    "gear",
    "gear_type",
    "sampling instrument",
    "sampling instrument name",
    "gear name",
    "samplinginstrument",
    "samplingdevice",
}

mt = obis_df["measurementType"].astype("string").str.strip().str.lower()
gear_rows = obis_df[mt.isin(targets)].copy()
gear_rows["measurementValue_str"] = gear_rows["measurementValue"].astype("string").str.strip()

# one value per id (first non-null, non-empty)
id_to_gear = (
    gear_rows[gear_rows["measurementValue_str"].notna() & (gear_rows["measurementValue_str"] != "")]
    .drop_duplicates(subset=["id"], keep="first")
    .set_index("id")["measurementValue_str"]
)

empty_gear_mask = merged_df["gear"].isna() | (merged_df["gear"].astype("string").str.strip() == "")
mapped_gear = merged_df["sourceID"].astype("string").map(id_to_gear)

fill_mask = empty_gear_mask & mapped_gear.notna()
merged_df.loc[fill_mask, "gear"] = mapped_gear[fill_mask]

added = int(fill_mask.sum())
print(f"Added {added} gear entries from OBIS measurementValue.")

merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)
print("Saved All_data/merged_all_mof.parquet")

Added 11667 gear entries from OBIS measurementValue.
Saved All_data/merged_all_mof.parquet


## Sex

In [ ]:
# Fill sex in All_data/merged_all_mof.parquet from OBIS measurementValue when sex is empty

obis_df = pd.read_parquet("OBIS/mof_OBIS_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

# Filter for sex-related measurement types (case-insensitive)
target_types = {
    "gender of biological entity specified elsewhere",
    "sex",
    "gender",
}

sex_rows = obis_df[
    obis_df["measurementType"].astype("string").str.strip().str.lower().isin(target_types)
].copy()

sex_rows["measurementValue_str"] = sex_rows["measurementValue"].astype("string").str.strip()

# One value per id (first non-null, non-empty)
id_to_sex = (
    sex_rows[sex_rows["measurementValue_str"].notna() & (sex_rows["measurementValue_str"] != "")]
    .drop_duplicates(subset=["id"], keep="first")
    .set_index("id")["measurementValue_str"]
)

# Empty sex mask
empty_sex_mask = merged_df["sex"].isna() | (merged_df["sex"].astype("string").str.strip() == "")

# Map OBIS id -> merged sourceID
mapped_sex = merged_df["sourceID"].astype("string").map(id_to_sex)

# Fill
fill_mask = empty_sex_mask & mapped_sex.notna()
merged_df.loc[fill_mask, "sex"] = mapped_sex[fill_mask]

added = int(fill_mask.sum())
print(f"Added {added} sex entries from OBIS measurementValue.")

merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)
print("Saved All_data/merged_all_mof.parquet")

Added 532 sex entries from OBIS measurementValue.
Saved All_data/merged_all_mof.parquet


## Mesh size

In [ ]:
# Fill meshSize_mm in All_data/merged_all_mof.parquet from OBIS when measurementType indicates mesh size

obis = pd.read_parquet("OBIS/mof_OBIS_filtered.parquet")
merged = pd.read_parquet("All_data/merged_all_mof.parquet")

targets = {"lower.sieve.mesh.size", "lowersievemeshsize", "mesh size"}
mt = obis["measurementType"].astype("string").str.strip().str.lower()
mesh_rows = obis[mt.isin(targets)].copy()

# Extract numeric part of measurementValue
mesh_rows["measurementValue_num"] = (
    mesh_rows["measurementValue"]
    .astype("string")
    .str.extract(r"([0-9]+(?:[.,][0-9]+)?)")[0]
    .str.replace(",", ".", regex=False)
    .astype(float)
)

# One value per id (first valid numeric)
id_to_mesh = (
    mesh_rows.dropna(subset=["measurementValue_num"])
    .drop_duplicates(subset=["id"], keep="first")
    .set_index("id")["measurementValue_num"]
)

mapped_mesh = merged["sourceID"].astype("string").map(id_to_mesh)

empty_mask = merged["meshSize_mm"].isna() | (merged["meshSize_mm"].astype("string").str.strip() == "")
fill_mask = empty_mask & mapped_mesh.notna()

merged.loc[fill_mask, "meshSize_mm"] = mapped_mesh[fill_mask]

added = int(fill_mask.sum())
merged.to_parquet("All_data/merged_all_mof.parquet", index=False)
print(f"Filled {added} meshSize_mm values from OBIS and saved All_data/merged_all_mof.parquet")

Filled 442 meshSize_mm values from OBIS and saved All_data/merged_all_mof.parquet


In [ ]:
# Fill meshSize_mm in All_data/merged_all_mof.parquet from OBIS sampling-net-mesh-size measurementValue
obis_df = pd.read_parquet("OBIS/mof_OBIS_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

targets = {
    "sampling net mesh size",
    "sampling.net.mesh.size",
    "samplingnetmeshsize",
    "size of sample collector (mesh) {mesh size}",
}

mt = obis_df["measurementType"].astype("string").str.strip().str.lower()
mesh_rows = obis_df[mt.isin(targets)].copy()

# extract numeric, convert comma->dot, to float
mesh_rows["measurementValue_num"] = (
    mesh_rows["measurementValue"]
    .astype("string")
    .str.extract(r"([0-9]+(?:[.,][0-9]+)?)")[0]
    .str.replace(",", ".", regex=False)
    .astype(float)
)

# UPDATED CONVERSION - Replace the division factor with correct conversion
mesh_rows["mesh_converted_mm"] = mesh_rows["measurementValue_num"] / 1000.0

# one value per id (first valid numeric)
id_to_mesh = (
    mesh_rows.dropna(subset=["mesh_converted_mm"])
    .drop_duplicates(subset=["id"], keep="first")
    .set_index("id")["mesh_converted_mm"]
)

mapped_mesh = merged_df["sourceID"].astype("string").map(id_to_mesh)

# OVERRIDE MODE: Fill all matching rows (even if already filled)
fill_mask = mapped_mesh.notna()

merged_df.loc[fill_mask, "meshSize_mm"] = mapped_mesh[fill_mask]

added = int(fill_mask.sum())
merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)
print(f"Filled {added} meshSize_mm values from OBIS sampling-net-mesh-size and saved All_data/merged_all_mof.parquet")

Filled 1715 meshSize_mm values from OBIS sampling-net-mesh-size and saved All_data/merged_all_mof.parquet


## Area

In [ ]:
# Fill areaSampled and areaSampledUnit from OBIS when measurementType == "area"

obis_df = pd.read_parquet("OBIS/mof_OBIS_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

# select area rows (case-insensitive)
area_rows = obis_df[obis_df["measurementType"].astype("string").str.strip().str.lower() == "area"].copy()
area_rows["measurementValue_str"] = area_rows["measurementValue"].astype("string").str.strip()

# try direct numeric parse, else extract first numeric token
area_rows["measurementValue_num"] = pd.to_numeric(
    area_rows["measurementValue_str"].str.replace(",", ".", regex=False), errors="coerce"
)
na_mask = area_rows["measurementValue_num"].isna()
area_rows.loc[na_mask, "measurementValue_num"] = (
    area_rows.loc[na_mask, "measurementValue_str"]
    .str.extract(r"([0-9]+(?:[.,][0-9]+)?)")[0]
    .str.replace(",", ".", regex=False)
    .astype(float)
)

# build maps (one value per id: first valid numeric)
id_map = (
    area_rows.dropna(subset=["measurementValue_num"])
    .drop_duplicates(subset=["id"], keep="first")
    .set_index("id")
)
id_to_value = id_map["measurementValue_num"]
id_to_unit = id_map["measurementUnit"].astype("string")

# masks for empty fields in merged_df
empty_area_mask = merged_df["areaSampled"].isna() | (merged_df["areaSampled"].astype("string").str.strip() == "")
empty_unit_mask = merged_df["areaSampledUnit"].isna() | (merged_df["areaSampledUnit"].astype("string").str.strip() == "")

# map OBIS id -> merged sourceID
mapped_value = merged_df["sourceID"].astype("string").map(id_to_value)
mapped_unit = merged_df["sourceID"].astype("string").map(id_to_unit)

# fill
fill_value_mask = empty_area_mask & mapped_value.notna()
fill_unit_mask = empty_unit_mask & mapped_unit.notna()

merged_df.loc[fill_value_mask, "areaSampled"] = mapped_value[fill_value_mask]
merged_df.loc[fill_unit_mask, "areaSampledUnit"] = mapped_unit[fill_unit_mask]

added_values = int(fill_value_mask.sum())
added_units = int(fill_unit_mask.sum())
print(f"Added {added_values} areaSampled values from OBIS measurementValue.")
print(f"Added {added_units} areaSampledUnit values from OBIS measurementUnit.")

merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)
print("Saved All_data/merged_all_mof.parquet")

Added 1780 areaSampled values from OBIS measurementValue.
Added 1780 areaSampledUnit values from OBIS measurementUnit.
Saved All_data/merged_all_mof.parquet


## Depth

In [ ]:
# Fill depthB_m from OBIS measurementValue when measurementType == depth_end
obis_df = pd.read_parquet("OBIS/mof_OBIS_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

# Filter depth_end rows (case-insensitive)
depth_rows = obis_df[
    obis_df["measurementType"].astype("string").str.strip().str.lower() == "depth_end"
].copy()

# Parse measurementValue as numeric
depth_rows["measurementValue_num"] = pd.to_numeric(
    depth_rows["measurementValue"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)

# One value per id (first valid numeric)
id_to_depth_end = (
    depth_rows.dropna(subset=["measurementValue_num"])
    .drop_duplicates(subset=["id"], keep="first")
    .set_index("id")["measurementValue_num"]
)

# Empty depthB_m mask
empty_depth_mask = merged_df["depthB_m"].isna() | (
    merged_df["depthB_m"].astype("string").str.strip() == ""
)

# Map OBIS id -> merged sourceID
mapped_depth_end = merged_df["sourceID"].astype("string").map(id_to_depth_end)

# Fill depthB_m and depthBType
fill_mask = empty_depth_mask & mapped_depth_end.notna()
merged_df.loc[fill_mask, "depthB_m"] = mapped_depth_end[fill_mask]
merged_df.loc[fill_mask, "depthBType"] = "end"

added_count = int(fill_mask.sum())
print(f"Added {added_count} depthB_m values from OBIS measurementType='depth_end'.")

merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)
print("Saved All_data/merged_all_mof.parquet")

Added 0 depthB_m values from OBIS measurementType='depth_end'.
Saved All_data/merged_all_mof.parquet


In [ ]:
# Fill depthA_m from OBIS measurementValue when measurementType == sampling_depth
obis_df = pd.read_parquet("OBIS/mof_OBIS_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

# Filter sampling_depth rows (case-insensitive)
depth_rows = obis_df[
    obis_df["measurementType"].astype("string").str.strip().str.lower() == "sampling_depth"
].copy()

# Parse measurementValue as numeric
depth_rows["measurementValue_num"] = pd.to_numeric(
    depth_rows["measurementValue"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)

# One value per id (first valid numeric)
id_to_sampling_depth = (
    depth_rows.dropna(subset=["measurementValue_num"])
    .drop_duplicates(subset=["id"], keep="first")
    .set_index("id")["measurementValue_num"]
)

# Normalize depthA_m before filling so numeric values can be written safely
merged_df["depthA_m"] = pd.to_numeric(merged_df["depthA_m"], errors="coerce")
empty_depth_mask = merged_df["depthA_m"].isna()

# Map OBIS id -> merged sourceID
mapped_sampling_depth = merged_df["sourceID"].astype("string").map(id_to_sampling_depth)

# Fill depthA_m and depthAType
fill_mask = empty_depth_mask & mapped_sampling_depth.notna()
merged_df.loc[fill_mask, "depthA_m"] = mapped_sampling_depth[fill_mask]
merged_df.loc[fill_mask, "depthAType"] = "unknown"

added_count = int(fill_mask.sum())
print(f"Added {added_count} depthA_m values from OBIS measurementType='sampling_depth'.")

merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)
print("Saved All_data/merged_all_mof.parquet")

Added 290 depthA_m values from OBIS measurementType='sampling_depth'.
Saved All_data/merged_all_mof.parquet


In [ ]:
# Fill depthA_m from OBIS measurementValue when measurementType == sampling_depth_max
obis_df = pd.read_parquet("OBIS/mof_OBIS_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

# Filter sampling_depth_max rows (case-insensitive)
depth_rows = obis_df[
    obis_df["measurementType"].astype("string").str.strip().str.lower() == "sampling_depth_max"
].copy()

# Parse measurementValue as numeric
depth_rows["measurementValue_num"] = pd.to_numeric(
    depth_rows["measurementValue"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)

# One value per id (first valid numeric)
id_to_sampling_depth = (
    depth_rows.dropna(subset=["measurementValue_num"])
    .drop_duplicates(subset=["id"], keep="first")
    .set_index("id")["measurementValue_num"]
)

# Normalize depthA_m before filling so numeric values can be written safely
merged_df["depthA_m"] = pd.to_numeric(merged_df["depthA_m"], errors="coerce")
empty_depth_mask = merged_df["depthA_m"].isna()

# Map OBIS id -> merged sourceID
mapped_sampling_depth = merged_df["sourceID"].astype("string").map(id_to_sampling_depth)

# Fill depthA_m and depthAType
fill_mask = empty_depth_mask & mapped_sampling_depth.notna()
merged_df.loc[fill_mask, "depthA_m"] = mapped_sampling_depth[fill_mask]
merged_df.loc[fill_mask, "depthAType"] = "maximum"

added_count = int(fill_mask.sum())
print(f"Added {added_count} depthA_m values from OBIS measurementType='sampling_depth_max'.")

merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)
print("Saved All_data/merged_all_mof.parquet")

Added 0 depthA_m values from OBIS measurementType='sampling_depth_max'.
Saved All_data/merged_all_mof.parquet


In [ ]:
# Fill depthB_m from OBIS measurementValue when measurementType == sampling_depth_in
obis_df = pd.read_parquet("OBIS/mof_OBIS_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

# Filter sampling_depth_min rows (case-insensitive)
depth_rows = obis_df[
    obis_df["measurementType"].astype("string").str.strip().str.lower() == "sampling_depth_min"
].copy()

# Parse measurementValue as numeric
depth_rows["measurementValue_num"] = pd.to_numeric(
    depth_rows["measurementValue"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)

# One value per id (first valid numeric)
id_to_sampling_depth = (
    depth_rows.dropna(subset=["measurementValue_num"])
    .drop_duplicates(subset=["id"], keep="first")
    .set_index("id")["measurementValue_num"]
)

# Normalize depthB_m before filling so numeric values can be written safely
merged_df["depthB_m"] = pd.to_numeric(merged_df["depthB_m"], errors="coerce")
empty_depth_mask = merged_df["depthB_m"].isna()

# Map OBIS id -> merged sourceID
mapped_sampling_depth = merged_df["sourceID"].astype("string").map(id_to_sampling_depth)

# Fill depthB_m and depthBType
fill_mask = empty_depth_mask & mapped_sampling_depth.notna()
merged_df.loc[fill_mask, "depthB_m"] = mapped_sampling_depth[fill_mask]
merged_df.loc[fill_mask, "depthBType"] = "minimum"

added_count = int(fill_mask.sum())
print(f"Added {added_count} depthB_m values from OBIS measurementType='sampling_depth_min'.")

merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)
print("Saved All_data/merged_all_mof.parquet")

Added 88 depthB_m values from OBIS measurementType='sampling_depth_min'.
Saved All_data/merged_all_mof.parquet


## Life stage

In [ ]:
# Fill lifeStage in All_data/merged_all_mof.parquet from OBIS measurementValue when lifeStage is empty

obis_df = pd.read_parquet("OBIS/mof_OBIS_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

# Filter for lifeStage-related measurement types (case-insensitive)
target_types = {
    "lifestage",
}

lifeStage_rows = obis_df[
    obis_df["measurementType"].astype("string").str.strip().str.lower().isin(target_types)
].copy()

lifeStage_rows["measurementValue_str"] = lifeStage_rows["measurementValue"].astype("string").str.strip()

# One value per id (first non-null, non-empty)
id_to_lifeStage = (
    lifeStage_rows[lifeStage_rows["measurementValue_str"].notna() & (lifeStage_rows["measurementValue_str"] != "")]
    .drop_duplicates(subset=["id"], keep="first")
    .set_index("id")["measurementValue_str"]
)

# Empty lifeStage mask
empty_lifeStage_mask = merged_df["lifeStage"].isna() | (merged_df["lifeStage"].astype("string").str.strip() == "")

# Map OBIS id -> merged sourceID
mapped_lifeStage = merged_df["sourceID"].astype("string").map(id_to_lifeStage)

# Fill
fill_mask = empty_lifeStage_mask & mapped_lifeStage.notna()
merged_df.loc[fill_mask, "lifeStage"] = mapped_lifeStage[fill_mask]

added = int(fill_mask.sum())
print(f"Added {added} lifeStage entries from OBIS measurementValue.")

merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)
print("Saved All_data/merged_all_mof.parquet")

Added 35 lifeStage entries from OBIS measurementValue.
Saved All_data/merged_all_mof.parquet


## Trawling distance

In [ ]:
obis_df = pd.read_parquet("OBIS/mof_OBIS_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

if "trawlingDistance_m" not in merged_df.columns:
    merged_df["trawlingDistance_m"] = pd.NA

target_types = {
    "trawling_distance",
    "trawling_distance_calculation",
    "length of sampling track",
    "trawling distance",
    "haul length in m",
}

obis_trawling = obis_df[
    obis_df["measurementType"].astype("string").str.strip().str.lower().isin(target_types)
].copy()

obis_trawling["measurementValue_num"] = pd.to_numeric(
    obis_trawling["measurementValue"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)

id_to_trawling = (
    obis_trawling.dropna(subset=["measurementValue_num"])
    .drop_duplicates(subset=["id"], keep="first")
    .set_index("id")["measurementValue_num"]
)

empty_mask = merged_df["trawlingDistance_m"].isna() | (
    merged_df["trawlingDistance_m"].astype("string").str.strip() == ""
)

mapped_trawling = merged_df["sourceID"].astype("string").map(id_to_trawling)
fill_mask = empty_mask & mapped_trawling.notna()

merged_df.loc[fill_mask, "trawlingDistance_m"] = mapped_trawling[fill_mask]

added = int(fill_mask.sum())
merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)

print(f"Added {added} trawlingDistance_m values.")
print("Saved All_data/merged_all_mof.parquet")

Added 1483 trawlingDistance_m values.
Saved All_data/merged_all_mof.parquet


OBIS/mof_OBIS_filtered.parquet and All_data/merged_all_mof.parquet are connected with the column id in the first file and the column sourceID in the second file. whenever measurementType in OBIS/mof_OBIS_filtered.parquet equals ABUNDNR, individualCount, count, observedindividualCount, taxon abundance, i want you to enter the value in the measurementValue column to the individualCount column in All_data/merged_all_mof.parquet (only when the latter column is empty). save parquet file with the same name and count how many entries were added.


OBIS/mof_OBIS_filtered.parquet and merged_all_mof.parquet are connected with the column id in the first file and the column sourceID in the second file. whenever measurementType in OBIS/mof_OBIS_filtered.parquet equals Biomass, Mass, Total macrofaunal biomass, Wet weight biomass (in assayed sample) of biological entity specified elsewhere, Wet weight biomass of biological entity specified elsewhere per unit area of the bed, biomass,  weight_flesh, weight_whole, wet weight,  i want you to enter the value in the measurementValue column to the individualCount column in All_data/merged_all_mof.parquet (only when the latter column is empty). also add the corresponding entries from measurementUnit to biomassUnit. save parquet file with the same name and count how many entries were added.

measurementType_unique_manual in OBIS folder

OBIS/mof_OBIS_filtered.parquet and merged_all_mof.parquet are connected with the column id in the first file and the column sourceID in the second file. whenever measurementType in OBIS/mof_OBIS_filtered.parquet equals depth_end, i want you to enter the value in the measurementValue column to the depthB_m column in All_data/merged_all_mof.parquet (only when the latter column is empty). also add "end" to the column depthBType whenever you add these new entries in the depthB_m column. save parquet file with the same name and count how many entries were added.

# GBIF MoF

## Filter and merge MoF

In [ ]:
# Load merged MOF data
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

# Load verbatim measurement file (tab-delimited)
input_file = r"GBIF\west_data\verbatim\dwc_measurementorfact.txt"
df_meas = pd.read_csv(input_file, sep="\t", low_memory=False)

# Filter by matching gbifID (normalize types to avoid mismatches)
meas_ids = df_meas["gbifid"].astype(str).str.strip()
gbif_ids = set(merged_df["sourceID"].astype(str).str.strip())
filtered = df_meas[meas_ids.isin(gbif_ids)]

# Save filtered CSV in the same folder
output_file = r"GBIF\west_data\verbatim\dwc_measurementorfact_filtered.csv"
filtered.to_csv(output_file, index=False)

print(f"Saved filtered file: {output_file} with {len(filtered)} rows")

Saved filtered file: GBIF\west_data\verbatim\dwc_measurementorfact_filtered.csv with 3307 rows


In [ ]:
# Load merged MOF data
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

# Load verbatim measurement file (tab-delimited)
input_file = r"GBIF\east_data\verbatim\dwc_measurementorfact.txt"
df_meas = pd.read_csv(input_file, sep="\t", low_memory=False)

# Filter by matching gbifID (normalize types to avoid mismatches)
meas_ids = df_meas["gbifid"].astype(str).str.strip()
gbif_ids = set(merged_df["sourceID"].astype(str).str.strip())
filtered = df_meas[meas_ids.isin(gbif_ids)]

# Save filtered CSV in the same folder
output_file = r"GBIF\east_data\verbatim\dwc_measurementorfact_filtered.csv"
filtered.to_csv(output_file, index=False)

print(f"Saved filtered file: {output_file} with {len(filtered)} rows")

Saved filtered file: GBIF\east_data\verbatim\dwc_measurementorfact_filtered.csv with 231156 rows


In [ ]:
# Combine GBIF east/west filtered measurement files and save as parquet
west = pd.read_csv(r"GBIF\west_data\verbatim\dwc_measurementorfact_filtered.csv", low_memory=False)
east = pd.read_csv(r"GBIF\east_data\verbatim\dwc_measurementorfact_filtered.csv", low_memory=False)

mof_gbif = pd.concat([west, east], ignore_index=True).drop_duplicates()
out_path = Path("GBIF") / "mof_gbif_filtered.parquet"
mof_gbif.to_parquet(out_path, index=False)

print(f"Saved {out_path} with {len(mof_gbif)} rows")

Saved GBIF\mof_gbif_filtered.parquet with 234463 rows


In [ ]:
# Also save unique measurementType + measurementUnit pairs from the filtered data
unique_pairs = (
    mof_gbif[["measurementtype", "measurementunit"]]
    .dropna(subset=["measurementtype"])  # require a value
    .drop_duplicates()
    .sort_values(["measurementtype", "measurementunit"], na_position="last")
    .reset_index(drop=True)
)
unique_out = r"GBIF\mof_measurement_types.csv"
unique_pairs.to_csv(unique_out, index=False, header=["measurementType", "measurementUnit"])


print(f"Saved {len(unique_pairs)} unique measurementType/measurementUnit pairs to {unique_out}")

Saved 154 unique measurementType/measurementUnit pairs to GBIF\mof_measurement_types.csv


## Filter and merge merged_GBIF.parquet

In [ ]:
# Filter GBIF records to keep only gbifID values present in All_data/merged_all_mof.parquet

mof_df = pd.read_parquet("All_data/merged_all_mof.parquet")
gbif_df = pd.read_parquet("GBIF/merged_GBIF.parquet")

valid_ids = set(mof_df["sourceID"].astype("string").str.strip())
gbif_ids = gbif_df["gbifID"].astype("string").str.strip()

gbif_filtered = gbif_df[gbif_ids.isin(valid_ids)].copy()

removed_count = len(gbif_df) - len(gbif_filtered)
out_path = Path("GBIF") / "merged_GBIF_filtered.parquet"
gbif_filtered.to_parquet(out_path, index=False)

print(f"Removed {removed_count} rows")
print(f"Saved {out_path} with {len(gbif_filtered)} rows")

Removed 225091 rows
Saved GBIF\merged_GBIF_filtered.parquet with 2363384 rows


In [ ]:
# Get unique organismQuantityType values from GBIF filtered data
unique_organism_quantity_type = (
    gbif_filtered["organismQuantityType"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

output_path = Path("GBIF") / "organismquantitytype.csv"
unique_organism_quantity_type.to_csv(output_path, index=False, header=["organismQuantityType"])

print(f"Saved {len(unique_organism_quantity_type)} unique organismQuantityType values to {output_path}")

Saved 144 unique organismQuantityType values to GBIF\organismquantitytype.csv


## individualCount

In [ ]:
# Fill individualCount, areaSampled, and areaSampledUnit from GBIF abundance measurements

# Load data
gbif_df = pd.read_parquet("GBIF/mof_gbif_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

# Find the relevant column names in GBIF output
mt_col = next((c for c in gbif_df.columns if c.lower() == "measurementtype"), "measurementtype")
mv_col = next((c for c in gbif_df.columns if c.lower() == "measurementvalue"), "measurementvalue")
id_col = next((c for c in gbif_df.columns if c.lower() == "gbifid"), "gbifid")

# Filter abundance rows
gbif_abund = gbif_df[gbif_df[mt_col].astype("string").str.strip().str.lower() == "abundance"].copy()

# Parse measurementValue as numeric
gbif_abund["measurementValue_num"] = pd.to_numeric(
    gbif_abund[mv_col].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)

# Normalize the join key to string before indexing so it matches merged_df["sourceID"]
gbif_abund[id_col] = gbif_abund[id_col].astype("string").str.strip()

# Keep one row per id (first valid numeric value)
id_map = (
    gbif_abund.dropna(subset=["measurementValue_num"])
    .drop_duplicates(subset=[id_col], keep="first")
    .set_index(id_col)
)

id_to_value = id_map["measurementValue_num"]

# Empty masks
empty_ic_mask = merged_df["individualCount"].isna() | (merged_df["individualCount"] == -999)
empty_area_mask = merged_df["areaSampled"].isna() | (merged_df["areaSampled"].astype("string").str.strip() == "")
empty_unit_mask = merged_df["areaSampledUnit"].isna() | (merged_df["areaSampledUnit"].astype("string").str.strip() == "")

# Map GBIF id -> merged sourceID
mapped_value = merged_df["sourceID"].astype("string").str.strip().map(id_to_value)

# Fill
fill_ic_mask = empty_ic_mask & mapped_value.notna()
fill_area_mask = empty_area_mask & mapped_value.notna()
fill_unit_mask = empty_unit_mask & mapped_value.notna()

merged_df.loc[fill_ic_mask, "individualCount"] = mapped_value[fill_ic_mask]
merged_df.loc[fill_area_mask, "areaSampled"] = 100.0
merged_df.loc[fill_unit_mask, "areaSampledUnit"] = "m2"

added = int(fill_ic_mask.sum())
print(f"Added {added} individualCount entries from GBIF abundance.")
print(f" - {int(fill_area_mask.sum())} areaSampled values set to 100.0")
print(f" - {int(fill_unit_mask.sum())} areaSampledUnit values set to m2")

merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)
print("Saved All_data/merged_all_mof.parquet")

Added 108310 individualCount entries from GBIF abundance.
 - 108310 areaSampled values set to 100.0
 - 108310 areaSampledUnit values set to m2
Saved All_data/merged_all_mof.parquet


In [ ]:
# Fill individualCount from GBIF measurementValue when measurementType == "Count"

gbif_df = pd.read_parquet("GBIF/mof_gbif_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

# Case-insensitive column lookup
mt_col = next(c for c in gbif_df.columns if c.lower() == "measurementtype")
mv_col = next(c for c in gbif_df.columns if c.lower() == "measurementvalue")
id_col = next(c for c in gbif_df.columns if c.lower() == "gbifid")

# Ensure target column exists
if "individualCount" not in merged_df.columns:
    merged_df["individualCount"] = pd.NA

# Filter target measurement type
rows = gbif_df[
    gbif_df[mt_col].astype("string").str.strip().str.lower() == "count"
].copy()

rows["measurementValue_str"] = rows[mv_col].astype("string").str.strip()
rows[id_col] = rows[id_col].astype("string").str.strip()

# One value per gbifid (first non-empty)
id_to_effort = (
    rows[rows["measurementValue_str"].notna() & (rows["measurementValue_str"] != "")]
    .drop_duplicates(subset=[id_col], keep="first")
    .set_index(id_col)["measurementValue_str"]
)

# Fill only empty individualCount
empty_mask = merged_df["individualCount"].isna() | (
    merged_df["individualCount"].astype("string").str.strip() == ""
)

mapped_effort_num = pd.to_numeric(
    mapped_effort.astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)
fill_mask = empty_mask & mapped_effort_num.notna()
merged_df.loc[fill_mask, "individualCount"] = mapped_effort_num[fill_mask]

added = int(fill_mask.sum())
merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)

print(f"Added {added} individualCount values from GBIF measurementType='Count'.")
print("Saved All_data/merged_all_mof.parquet")

Added 0 individualCount values from GBIF measurementType='Count'.
Saved All_data/merged_all_mof.parquet


In [ ]:
# Fill individualCount from GBIF organismQuantity when organismQuantityType == "individuals/10cm^2"

gbif_df = pd.read_parquet("GBIF/merged_GBIF_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

# Case-insensitive column lookup
oqt_col = next(c for c in gbif_df.columns if c.lower() == "organismquantitytype")
oq_col = next(c for c in gbif_df.columns if c.lower() == "organismquantity")
id_col = next(c for c in gbif_df.columns if c.lower() == "gbifid")

# Normalize
gbif_df[id_col] = gbif_df[id_col].astype("string").str.strip()
gbif_df["organismQuantityType_norm"] = gbif_df[oqt_col].astype("string").str.strip().str.lower()
gbif_df["organismQuantity_num"] = pd.to_numeric(
    gbif_df[oq_col].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)

# Filter target rows
gbif_qty = gbif_df[gbif_df["organismQuantityType_norm"] == "individuals/10cm^2"].copy()

# One value per gbifID
id_to_qty = (
    gbif_qty.dropna(subset=["organismQuantity_num"])
    .drop_duplicates(subset=[id_col], keep="first")
    .set_index(id_col)["organismQuantity_num"]
)

# Empty individualCount mask
empty_ic_mask = merged_df["individualCount"].isna() | (
    merged_df["individualCount"].astype("string").str.strip().isin(["", "-999"])
)

mapped_qty = merged_df["sourceID"].astype("string").str.strip().map(id_to_qty)
fill_mask = empty_ic_mask & mapped_qty.notna()

# Fill fields
merged_df.loc[fill_mask, "individualCount"] = mapped_qty[fill_mask]
merged_df.loc[fill_mask, "areaSampled"] = 10.0
merged_df.loc[fill_mask, "areaSampledUnit"] = "cm2"

added = int(fill_mask.sum())
merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)

print(f"Added {added} individualCount values from GBIF organismQuantityType='individuals/10cm^2'.")
print("Saved All_data/merged_all_mof.parquet")

Added 1169 individualCount values from GBIF organismQuantityType='individuals/10cm^2'.
Saved All_data/merged_all_mof.parquet


In [ ]:
# Fill individualCount from GBIF organismQuantity when organismQuantityType matches specified area types.
gbif_df = pd.read_parquet("GBIF/merged_GBIF_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

# case-insensitive column lookup
oqt_col = next((c for c in gbif_df.columns if c.lower() == "organismquantitytype"), "organismQuantityType")
oq_col = next((c for c in gbif_df.columns if c.lower() == "organismquantity"), "organismQuantity")
id_col = next((c for c in gbif_df.columns if c.lower() == "gbifid"), "gbifID")

# target types (normalized)
target_types = {"individuals per square metre", "m2", "square metre"}

gbif_df["organismQuantityType_norm"] = gbif_df[oqt_col].astype("string").str.strip().str.lower()
gbif_df["organismQuantity_num"] = pd.to_numeric(
    gbif_df[oq_col].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)

gbif_df[id_col] = gbif_df[id_col].astype("string").str.strip()

gbif_qty = gbif_df[gbif_df["organismQuantityType_norm"].isin(target_types)].copy()

id_to_qty = (
    gbif_qty.dropna(subset=["organismQuantity_num"])
    .drop_duplicates(subset=[id_col], keep="first")
    .set_index(id_col)["organismQuantity_num"]
)

# ensure individualCount column exists
if "individualCount" not in merged_df.columns:
    merged_df["individualCount"] = pd.NA

empty_ic_mask = merged_df["individualCount"].isna() | (
    merged_df["individualCount"].astype("string").str.strip().isin(["", "-999"])
)

mapped_qty = merged_df["sourceID"].astype("string").str.strip().map(id_to_qty)
fill_mask = empty_ic_mask & mapped_qty.notna()

# assign individualCount
merged_df.loc[fill_mask, "individualCount"] = mapped_qty[fill_mask]

# update areaSampled (add 1) and set areaSampledUnit to "m2" for rows where we filled individualCount
# compute existing numeric area (treat non-numeric as 0)
area_existing = pd.to_numeric(merged_df.loc[fill_mask, "areaSampled"], errors="coerce").fillna(0)
merged_df.loc[fill_mask, "areaSampled"] = area_existing + 1.0
merged_df.loc[fill_mask, "areaSampledUnit"] = "m2"

added = int(fill_mask.sum())
merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)
print(f"Added {added} individualCount entries (and updated areaSampled/areaSampledUnit).")

Added 615 individualCount entries (and updated areaSampled/areaSampledUnit).


In [ ]:
gbif_df = pd.read_parquet("GBIF/merged_GBIF_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

target_types = {"abundance", "individuals", "individuals per area"}

gbif_df["gbifID"] = gbif_df["gbifID"].astype("string").str.strip()
gbif_df["organismQuantityType_norm"] = gbif_df["organismQuantityType"].astype("string").str.strip().str.lower()
gbif_df["organismQuantity_num"] = pd.to_numeric(
    gbif_df["organismQuantity"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)

gbif_qty = gbif_df[
    gbif_df["organismQuantityType_norm"].isin(target_types)
].copy()

id_to_qty = (
    gbif_qty.dropna(subset=["organismQuantity_num"])
    .drop_duplicates(subset=["gbifID"], keep="first")
    .set_index("gbifID")["organismQuantity_num"]
)

empty_ic_mask = merged_df["individualCount"].isna() | (
    merged_df["individualCount"].astype("string").str.strip().isin(["", "-999"])
)

mapped_qty = merged_df["sourceID"].astype("string").str.strip().map(id_to_qty)
fill_mask = empty_ic_mask & mapped_qty.notna()

merged_df.loc[fill_mask, "individualCount"] = mapped_qty[fill_mask]

added = int(fill_mask.sum())
merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)

print(f"Added {added} individualCount values from GBIF organismQuantity.")
print("Saved All_data/merged_all_mof.parquet")

Added 31771 individualCount values from GBIF organismQuantity.
Saved All_data/merged_all_mof.parquet


## Biomass

In [ ]:
# Fill biomass, areaSampled, and areaSampledUnit from GBIF biomass measurements

# Load data
gbif_df = pd.read_parquet("GBIF/mof_gbif_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

# Find the relevant column names in GBIF output
mt_col = next((c for c in gbif_df.columns if c.lower() == "measurementtype"), "measurementtype")
mv_col = next((c for c in gbif_df.columns if c.lower() == "measurementvalue"), "measurementvalue")
id_col = next((c for c in gbif_df.columns if c.lower() == "gbifid"), "gbifid")

# Filter biomass rows
gbif_biomass = gbif_df[gbif_df[mt_col].astype("string").str.strip().str.lower() == "biomass"].copy()

# Parse measurementValue as numeric
gbif_biomass["measurementValue_num"] = pd.to_numeric(
    gbif_biomass[mv_col].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)

# Normalize the join key to string before indexing so it matches merged_df["sourceID"]
gbif_biomass[id_col] = gbif_biomass[id_col].astype("string").str.strip()

# Keep one row per id (first valid numeric value)
id_map = (
    gbif_biomass.dropna(subset=["measurementValue_num"])
    .drop_duplicates(subset=[id_col], keep="first")
    .set_index(id_col)
)

id_to_value = id_map["measurementValue_num"]

# Empty masks
empty_ic_mask = merged_df["biomass"].isna() | (merged_df["biomass"] == -999)
empty_biounit_mask = merged_df["biomassUnit"].isna() | (merged_df["biomassUnit"].astype("string").str.strip() == "")
empty_area_mask = merged_df["areaSampled"].isna() | (merged_df["areaSampled"].astype("string").str.strip() == "")
empty_unit_mask = merged_df["areaSampledUnit"].isna() | (merged_df["areaSampledUnit"].astype("string").str.strip() == "")

# Map GBIF id -> merged sourceID
mapped_value = merged_df["sourceID"].astype("string").str.strip().map(id_to_value)

# Fill
fill_ic_mask = empty_ic_mask & mapped_value.notna()
fill_area_mask = empty_area_mask & mapped_value.notna()
fill_unit_mask = empty_unit_mask & mapped_value.notna()
fill_biounit_mask = empty_biounit_mask & mapped_value.notna()

merged_df.loc[fill_ic_mask, "biomass"] = mapped_value[fill_ic_mask]
merged_df.loc[fill_area_mask, "areaSampled"] = 100.0
merged_df.loc[fill_unit_mask, "areaSampledUnit"] = "m2"
merged_df.loc[fill_biounit_mask, "biomassUnit"] = "g"

added = int(fill_ic_mask.sum())
print(f"Added {added} biomass entries from GBIF biomass.")
print(f" - {int(fill_area_mask.sum())} areaSampled values set to 100.0")
print(f" - {int(fill_unit_mask.sum())} areaSampledUnit values set to m2")
print(f" - {int(fill_biounit_mask.sum())} biomassUnit values set to g")

merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)
print("Saved All_data/merged_all_mof.parquet")

Added 110657 biomass entries from GBIF biomass.
 - 3708 areaSampled values set to 100.0
 - 3708 areaSampledUnit values set to m2
 - 110657 biomassUnit values set to g
Saved All_data/merged_all_mof.parquet


In [ ]:
# Fill biomass, areaSampled, and areaSampledUnit from GBIF biomass measurements

# Load data
gbif_df = pd.read_parquet("GBIF/mof_gbif_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

# Find the relevant column names in GBIF output
mt_col = next((c for c in gbif_df.columns if c.lower() == "measurementtype"), "measurementtype")
mv_col = next((c for c in gbif_df.columns if c.lower() == "measurementvalue"), "measurementvalue")
id_col = next((c for c in gbif_df.columns if c.lower() == "gbifid"), "gbifid")

# Filter biomass rows
gbif_biomass = gbif_df[gbif_df[mt_col].astype("string").str.strip().str.lower() == "weight_whole"].copy()

# Parse measurementValue as numeric
gbif_biomass["measurementValue_num"] = pd.to_numeric(
    gbif_biomass[mv_col].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)

# Normalize the join key to string before indexing so it matches merged_df["sourceID"]
gbif_biomass[id_col] = gbif_biomass[id_col].astype("string").str.strip()

# Keep one row per id (first valid numeric value)
id_map = (
    gbif_biomass.dropna(subset=["measurementValue_num"])
    .drop_duplicates(subset=[id_col], keep="first")
    .set_index(id_col)
)

id_to_value = id_map["measurementValue_num"]

# Empty masks
empty_ic_mask = merged_df["biomass"].isna() | (merged_df["biomass"] == -999)
empty_biounit_mask = merged_df["biomassUnit"].isna() | (merged_df["biomassUnit"].astype("string").str.strip() == "")

# Map GBIF id -> merged sourceID
mapped_value = merged_df["sourceID"].astype("string").str.strip().map(id_to_value)

# Fill
fill_ic_mask = empty_ic_mask & mapped_value.notna()
fill_area_mask = empty_area_mask & mapped_value.notna()
fill_unit_mask = empty_unit_mask & mapped_value.notna()
fill_biounit_mask = empty_biounit_mask & mapped_value.notna()

merged_df.loc[fill_ic_mask, "biomass"] = mapped_value[fill_ic_mask]
merged_df.loc[fill_biounit_mask, "biomassUnit"] = "g"

added = int(fill_ic_mask.sum())
print(f"Added {added} biomass entries from GBIF biomass.")
print(f" - {int(fill_biounit_mask.sum())} biomassUnit values set to g")

merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)
print("Saved All_data/merged_all_mof.parquet")

Added 20 biomass entries from GBIF biomass.
 - 20 biomassUnit values set to g
Saved All_data/merged_all_mof.parquet


In [ ]:
# Fill biomass from GBIF organismQuantity when organismQuantityType == "gram"

gbif_df = pd.read_parquet("GBIF/merged_GBIF_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

oqt_col = next(c for c in gbif_df.columns if c.lower() == "organismquantitytype")
oq_col = next(c for c in gbif_df.columns if c.lower() == "organismquantity")
id_col = next(c for c in gbif_df.columns if c.lower() == "gbifid")

gbif_df[id_col] = gbif_df[id_col].astype("string").str.strip()
gbif_df["organismQuantityType_norm"] = gbif_df[oqt_col].astype("string").str.strip().str.lower()
gbif_df["organismQuantity_num"] = pd.to_numeric(
    gbif_df[oq_col].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)

gbif_gram = gbif_df[gbif_df["organismQuantityType_norm"] == "gram"].copy()

id_to_biomass = (
    gbif_gram.dropna(subset=["organismQuantity_num"])
    .drop_duplicates(subset=[id_col], keep="first")
    .set_index(id_col)["organismQuantity_num"]
)

empty_biomass_mask = merged_df["biomass"].isna() | (
    merged_df["biomass"].astype("string").str.strip().isin(["", "-999"])
)

mapped_biomass = merged_df["sourceID"].astype("string").str.strip().map(id_to_biomass)
fill_mask = empty_biomass_mask & mapped_biomass.notna()

merged_df.loc[fill_mask, "biomass"] = mapped_biomass[fill_mask]
merged_df.loc[fill_mask, "biomassUnit"] = "g"

added = int(fill_mask.sum())
merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)

print(f"Added {added} biomass values from GBIF organismQuantityType='gram'.")
print("Saved All_data/merged_all_mof.parquet")

Added 143 biomass values from GBIF organismQuantityType='gram'.
Saved All_data/merged_all_mof.parquet


## Depth

In [ ]:
# Fill depthA_m from GBIF measurementValue when measurementType == depth max
gbif_df = pd.read_parquet("GBIF/mof_gbif_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

# Case-insensitive column lookup
mt_col = next(c for c in gbif_df.columns if c.lower() == "measurementtype")
mv_col = next(c for c in gbif_df.columns if c.lower() == "measurementvalue")
id_col = next(c for c in gbif_df.columns if c.lower() == "gbifid")

# Filter Depth max rows (case-insensitive)
depth_rows = gbif_df[
    gbif_df[mt_col].astype("string").str.strip().str.lower() == "depth max"
].copy()

# Parse measurementValue as numeric
depth_rows["measurementvalue_num"] = pd.to_numeric(
    depth_rows[mv_col].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)

# One value per GBIF id (first valid numeric)
depth_rows[id_col] = depth_rows[id_col].astype("string").str.strip()
id_to_depth_max = (
    depth_rows.dropna(subset=["measurementvalue_num"])
    .drop_duplicates(subset=[id_col], keep="first")
    .set_index(id_col)["measurementvalue_num"]
)

# Empty depthA_m mask
empty_depth_mask = merged_df["depthA_m"].isna() | (
    merged_df["depthA_m"].astype("string").str.strip() == ""
)

# Map GBIF id -> merged sourceID
mapped_depth_max = merged_df["sourceID"].astype("string").str.strip().map(id_to_depth_max)

# Fill depthA_m and depthAType
fill_mask = empty_depth_mask & mapped_depth_max.notna()
merged_df.loc[fill_mask, "depthA_m"] = mapped_depth_max[fill_mask]
merged_df.loc[fill_mask, "depthAType"] = "maximum"

added_count = int(fill_mask.sum())
print(f"Added {added_count} depthA_m values from GBIF measurementType='Depth max'.")

merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)
print("Saved All_data/merged_all_mof.parquet")

Added 584 depthA_m values from GBIF measurementType='Depth max'.
Saved All_data/merged_all_mof.parquet


In [ ]:
# Fill depthB_m from GBIF measurementValue when measurementType == depth min
gbif_df = pd.read_parquet("GBIF/mof_gbif_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

# Case-insensitive column lookup
mt_col = next(c for c in gbif_df.columns if c.lower() == "measurementtype")
mv_col = next(c for c in gbif_df.columns if c.lower() == "measurementvalue")
id_col = next(c for c in gbif_df.columns if c.lower() == "gbifid")

# Filter Depth min rows (case-insensitive)
depth_rows = gbif_df[
    gbif_df[mt_col].astype("string").str.strip().str.lower() == "depth min"
].copy()

# Parse measurementValue as numeric
depth_rows["measurementvalue_num"] = pd.to_numeric(
    depth_rows[mv_col].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)

# One value per GBIF id (first valid numeric)
depth_rows[id_col] = depth_rows[id_col].astype("string").str.strip()
id_to_depth_min = (
    depth_rows.dropna(subset=["measurementvalue_num"])
    .drop_duplicates(subset=[id_col], keep="first")
    .set_index(id_col)["measurementvalue_num"]
)

# Empty depthB_m mask
empty_depth_mask = merged_df["depthB_m"].isna() | (
    merged_df["depthB_m"].astype("string").str.strip() == ""
)

# Map GBIF id -> merged sourceID
mapped_depth_min = merged_df["sourceID"].astype("string").str.strip().map(id_to_depth_min)

# Fill depthB_m and depthBType
fill_mask = empty_depth_mask & mapped_depth_min.notna()
merged_df.loc[fill_mask, "depthB_m"] = mapped_depth_min[fill_mask]
merged_df.loc[fill_mask, "depthBType"] = "minimum"

added_count = int(fill_mask.sum())
print(f"Added {added_count} depthB_m values from GBIF measurementType='Depth min'.")

merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)
print("Saved All_data/merged_all_mof.parquet")

Added 584 depthB_m values from GBIF measurementType='Depth min'.
Saved All_data/merged_all_mof.parquet


## samplingEffort

In [ ]:
# Fill samplingEffort from GBIF measurementValue when measurementType == "Sample collecting device"

gbif_df = pd.read_parquet("GBIF/mof_gbif_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

# Case-insensitive column lookup
mt_col = next(c for c in gbif_df.columns if c.lower() == "measurementtype")
mv_col = next(c for c in gbif_df.columns if c.lower() == "measurementvalue")
id_col = next(c for c in gbif_df.columns if c.lower() == "gbifid")

# Ensure target column exists
if "samplingEffort" not in merged_df.columns:
    merged_df["samplingEffort"] = pd.NA

# Filter target measurement type
rows = gbif_df[
    gbif_df[mt_col].astype("string").str.strip().str.lower() == "sample collecting device"
].copy()

rows["measurementValue_str"] = rows[mv_col].astype("string").str.strip()
rows[id_col] = rows[id_col].astype("string").str.strip()

# One value per gbifid (first non-empty)
id_to_effort = (
    rows[rows["measurementValue_str"].notna() & (rows["measurementValue_str"] != "")]
    .drop_duplicates(subset=[id_col], keep="first")
    .set_index(id_col)["measurementValue_str"]
)

# Fill only empty samplingEffort
empty_mask = merged_df["samplingEffort"].isna() | (
    merged_df["samplingEffort"].astype("string").str.strip() == ""
)

mapped_effort = merged_df["sourceID"].astype("string").str.strip().map(id_to_effort)
fill_mask = empty_mask & mapped_effort.notna()

merged_df.loc[fill_mask, "samplingEffort"] = mapped_effort[fill_mask]

added = int(fill_mask.sum())
merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)

print(f"Added {added} samplingEffort values from GBIF measurementType='Sample collecting device'.")
print("Saved All_data/merged_all_mof.parquet")

Added 579 samplingEffort values from GBIF measurementType='Sample collecting device'.
Saved All_data/merged_all_mof.parquet


# Save with the correct data types

In [ ]:
schema_path = Path("column_data_types.xlsx")
schema_df = pd.read_excel(schema_path, sheet_name="types")
schema_map = dict(zip(schema_df["column"], schema_df["final"]))


def coerce_column_to_schema(series, dtype_name):
    dtype_name = str(dtype_name).strip()

    if dtype_name == "string":
        return series.astype("string")
    if dtype_name == "float64":
        return pd.to_numeric(series, errors="coerce")
    if dtype_name == "Int64":
        return pd.to_numeric(series, errors="coerce").round().astype("Int64")
    if dtype_name == "boolean":
        normalized = series.astype("string").str.strip().str.lower()
        return normalized.map({"true": True, "false": False, "1": True, "0": False}).astype("boolean")

    return series


def enforce_schema_in_file(parquet_path):
    parquet_path = Path(parquet_path)
    if not parquet_path.exists():
        print(f"Skipped missing file: {parquet_path}")
        return

    frame = pd.read_parquet(parquet_path)

    for column_name, dtype_name in schema_map.items():
        if column_name in frame.columns:
            frame[column_name] = coerce_column_to_schema(frame[column_name], dtype_name)

    ordered_columns = [column for column in schema_df["column"] if column in frame.columns]
    ordered_columns.extend([column for column in frame.columns if column not in ordered_columns])
    frame = frame[ordered_columns]

    frame.to_parquet(parquet_path, index=False)
    print(f"Enforced schema for {parquet_path}")


for parquet_file in [
    "All_data/merged_all_mof.parquet",
    "All_data/merged_all_filtered.parquet",
]:
    enforce_schema_in_file(parquet_file)

Enforced schema for merged_all_mof.parquet
Enforced schema for All_data\merged_all_mof.parquet
Enforced schema for All_data\merged_all_filtered.parquet


Next steps: 
Deduplicate within and across databases
Look through gear types
GEBCO depth column (needed?)